# Lab 2 — Oxford Pet Semantic Segmentation
U-Net / ResNet34-UNet 訓練與推論

## 使用前準備
1. 執行階段 → 變更執行階段類型 → **GPU**
2. Google Drive 的 `MyDrive/lab2/` 裡要有 `saved_models/`、`nycu-2026-spring-dl-lab2-unet/`、`binary-semantic-segmentation-res-net-34-u-net/` 三個資料夾
3. 程式碼從 GitHub clone，**不需要上傳 src/**
4. 圖片資料集第一次執行時自動下載至 Colab 本地，**不需要上傳至 Drive**


## 1. 確認 GPU

In [ ]:
# ── 1. 確認 GPU ──────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


## 2. Clone Repo

In [ ]:
# ── 2. Clone repo（只拿程式碼，很快）(第一次做就好)────────────────────────────────
import os

if not os.path.exists('/content/NYCU-Deep-Learing'):
    !git clone https://github.com/InnisChen/NYCU-Deep-Learing /content/NYCU-Deep-Learing
else:
    print('Repo already cloned, pulling latest...')
    !git -C /content/NYCU-Deep-Learing pull

%cd /content/NYCU-Deep-Learing/lab2
import sys
sys.path.insert(0, 'src')
print(f'Working directory: {os.getcwd()}')

Cloning into '/content/NYCU-Deep-Learing'...
remote: Enumerating objects: 306, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 306 (delta 16), reused 35 (delta 15), pack-reused 264 (from 1)
Receiving objects: 100% (306/306), 889.28 KiB | 15.07 MiB/s, done.
Resolving deltas: 100% (158/158), done.
/content/NYCU-Deep-Learing/lab2
Working directory: /content/NYCU-Deep-Learing/lab2


## 3. 下載圖片


In [ ]:
# 拿picture (第一次就好)
from src.oxford_pet import download_dataset
download_dataset("dataset/oxford-iiit-pet")

Extracting images...
images done.
Extracting annotations...
annotations done.


## 4. 掛載 Google Drive

In [ ]:
# ── 3. 掛載 Google Drive ─────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 5. 確認資料集結構

In [ ]:
# ── 5. 確認資料集結構 ─────────────────────────────────────────────────
dataset_root = 'dataset/oxford-iiit-pet'
#DATA_PATH = '/content/drive/MyDrive/lab2/dataset/oxford-iiit-pet' #直接使用drive 內
print('Images :', len(os.listdir(f'{dataset_root}/images')), 'files')
print('Trimaps:', len(os.listdir(f'{dataset_root}/annotations/trimaps')), 'files')

for split_dir in ['nycu-2026-spring-dl-lab2-unet', 'binary-semantic-segmentation-res-net-34-u-net']:
    if os.path.exists(split_dir):
        files_found = os.listdir(split_dir)
        print(f'{split_dir}/: {files_found}')
    else:
        print(f'MISSING: {split_dir}/')

Images : 7393 files
Trimaps: 14780 files
nycu-2026-spring-dl-lab2-unet/: ['test_unet.txt', 'train.txt', 'val.txt']
binary-semantic-segmentation-res-net-34-u-net/: ['train.txt', 'val.txt', 'test_res_unet.txt']


## 6. 競賽一：UNet 訓練

In [ ]:
import shutil, os

SAVE_PATH   = '/content/saved_models'
BACKUP_PATH = '/content/drive/MyDrive/lab2/saved_models'

os.makedirs(SAVE_PATH, exist_ok=True)

# 從 Drive 把 epoch40 複製回本地，再設為 checkpoint（unet）
src = os.path.join(BACKUP_PATH, 'unet_epoch40.pth')
dst = os.path.join(SAVE_PATH,   'unet_checkpoint.pth')
shutil.copy(src, dst)
print(f"Copied: {src} → {dst}")

NameError: name 'SAVE_PATH' is not defined

In [ ]:
# ── 7. 設定訓練參數 ───────────────────────────────────────────────────
MODEL       = 'unet'
EPOCHS      = 200
BATCH_SIZE  = 32
LR          = 2e-4
NUM_WORKERS = 48
RESUME      = False
DATA_PATH   = 'dataset/oxford-iiit-pet'
SAVE_PATH   = '/content/saved_models'                      # 本地（快速寫入）
BACKUP_PATH = '/content/drive/MyDrive/lab2/saved_models'   # Drive（每 40 epoch 備份）
SPLIT_DIR   = 'nycu-2026-spring-dl-lab2-unet'

cmd = (f'python src/train.py'
       f' --model {MODEL}'
       f' --split_dir {SPLIT_DIR}'
       f' --data_path {DATA_PATH}'
       f' --save_path {SAVE_PATH}'
       f' --backup_path {BACKUP_PATH}'
       f' --epochs {EPOCHS}'
       f' --batch_size {BATCH_SIZE}'
       f' --learning_rate {LR}'
       f' --num_workers {NUM_WORKERS}')
if RESUME:
    cmd += ' --resume'
print('Command:', cmd)


Command: python src/train.py --model unet --split_dir nycu-2026-spring-dl-lab2-unet --data_path dataset/oxford-iiit-pet --save_path /content/saved_models --backup_path /content/drive/MyDrive/lab2/saved_models --epochs 200 --batch_size 32 --learning_rate 0.0002 --num_workers 48


In [ ]:
# ── 執行訓練（結果直接存進 Drive，不會因 session 結束消失）──────────
import os
os.makedirs(SAVE_PATH, exist_ok=True)
!{cmd}

Device: cuda
Train batches: 162, Valid batches: 24
Model: unet  |  Params: 31,031,745
Training:   0% 0/200 [00:00<?, ?it/s]Epoch   1/200  loss=0.6243  val_dice=0.4384
  → Best model saved (dice=0.4384)
Training:   0% 1/200 [00:52<2:52:41, 52.07s/it]Epoch   2/200  loss=0.5972  val_dice=0.4953
  → Best model saved (dice=0.4953)
Training:   1% 2/200 [01:41<2:45:51, 50.26s/it]Epoch   3/200  loss=0.5776  val_dice=0.4253
Training:   2% 3/200 [02:30<2:43:39, 49.84s/it]Epoch   4/200  loss=0.5029  val_dice=0.5750
  → Best model saved (dice=0.5750)
Training:   2% 4/200 [03:19<2:42:14, 49.67s/it]Epoch   5/200  loss=0.4312  val_dice=0.6814
  → Best model saved (dice=0.6814)
Training:   2% 5/200 [04:09<2:41:27, 49.68s/it]Epoch   6/200  loss=0.3839  val_dice=0.6879
  → Best model saved (dice=0.6879)
Training:   3% 6/200 [04:59<2:40:49, 49.74s/it]Epoch   7/200  loss=0.3380  val_dice=0.7640
  → Best model saved (dice=0.7640)
Training:   4% 7/200 [05:49<2:39:59, 49.74s/it]Epoch   8/200  loss=0.3095  va

## 7. 競賽一：UNet 推論（生成 submission.csv）

In [ ]:
# ── 8. 推論設定 ───────────────────────────────────────────────────────
MODEL      = 'unet'          # 與訓練時一致
DATA_PATH  = 'dataset/oxford-iiit-pet'
WEIGHT     = f'/content/drive/MyDrive/lab2/saved_models/{MODEL}_best.pth'
OUTPUT     = f'submission_{MODEL}.csv'
BATCH_SIZE = 32   #不需要跟訓練size一樣，只影響推論速度。  不要超過RAM限制就好

cmd_inf = (f'python src/inference.py'
           f' --model {MODEL}'
           f' --data_path {DATA_PATH}'
           f' --weight {WEIGHT}'
           f' --output {OUTPUT}'
           f' --batch_size {BATCH_SIZE}')
print('Command:', cmd_inf)

Command: python src/inference.py --model unet --data_path dataset/oxford-iiit-pet --weight /content/drive/MyDrive/lab2/saved_models/unet_best.pth --output submission_unet.csv --batch_size 32


In [ ]:
# ── 執行推論 ──────────────────────────────────────────────────────────
!{cmd_inf}

Device: cuda
Split dir: nycu-2026-spring-dl-lab2-unet
Loaded weights: /content/drive/MyDrive/lab2/saved_models/unet_best.pth
Test samples: 739
Inference: 100% 24/24 [02:18<00:00,  5.77s/it]
Saved 739 predictions → submission_unet.csv


In [ ]:
# ── 確認並下載 submission.csv ─────────────────────────────────────────
import pandas as pd
from google.colab import files

df = pd.read_csv(OUTPUT)
print(f'Submission shape: {df.shape}')
print(df.head())

# 存到 Drive
import shutil
shutil.copy(OUTPUT, f'/content/drive/MyDrive/lab2/{OUTPUT}')
print(f'Saved to Drive')


Submission shape: (739, 2)
                 image_id                                       encoded_mask
0        basset_hound_177  13566 6 13897 6 14225 9 14553 12 14880 16 1521...
1             keeshond_75  28416 1 28418 1 28616 1 28736 3 28923 23 29056...
2      wheaten_terrier_25  30673 17 31170 22 31670 25 32169 29 32669 31 3...
3               Sphynx_36  61347 26 61674 37 61999 48 62323 58 62656 58 6...
4  german_shorthaired_161  35092 15 35425 15 35754 23 36086 26 36416 30 3...
Saved to Drive


In [ ]:
# 下載到本機
files.download(OUTPUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. 競賽二：ResNet34-UNet 訓練（Binary Semantic Segmentation）

In [ ]:
import shutil, os

SAVE_PATH   = '/content/saved_models'
BACKUP_PATH = '/content/drive/MyDrive/lab2/saved_models'

os.makedirs(SAVE_PATH, exist_ok=True)

# 從 Drive 把 epoch40 複製回本地，再設為 checkpoint
src = os.path.join(BACKUP_PATH, 'resnet34_unet_epoch40.pth')
dst = os.path.join(SAVE_PATH,   'resnet34_unet_checkpoint.pth')
shutil.copy(src, dst)
print(f"Copied: {src} → {dst}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/lab2/saved_models/resnet34_unet_epoch40.pth'

In [ ]:
# ── 9. ResNet34-UNet 訓練設定 ────────────────────────────────────────
MODEL       = 'resnet34_unet'
EPOCHS      = 200
BATCH_SIZE  = 32         #32 1e-4    64 2e-4  128 4e-4
LR          = 2e-4
NUM_WORKERS = 48
RESUME      = False            # True = 從上次中斷繼續；False = 重新開始
DATA_PATH   = 'dataset/oxford-iiit-pet'
SAVE_PATH   = '/content/saved_models'                      # 本地（快速寫入）
BACKUP_PATH = '/content/drive/MyDrive/lab2/saved_models'   # Drive（每 40 epoch 備份）
SPLIT_DIR   = 'binary-semantic-segmentation-res-net-34-u-net'

cmd_r34_train = (f'python src/train.py'
                 f' --model {MODEL}'
                 f' --split_dir {SPLIT_DIR}'
                 f' --data_path {DATA_PATH}'
                 f' --save_path {SAVE_PATH}'
                 f' --backup_path {BACKUP_PATH}'
                 f' --epochs {EPOCHS}'
                 f' --batch_size {BATCH_SIZE}'
                 f' --learning_rate {LR}'
                 f' --num_workers {NUM_WORKERS}')
if RESUME:
    cmd_r34_train += ' --resume'
print('Command:', cmd_r34_train)


Command: python src/train.py --model resnet34_unet --split_dir binary-semantic-segmentation-res-net-34-u-net --data_path dataset/oxford-iiit-pet --save_path /content/saved_models --backup_path /content/drive/MyDrive/lab2/saved_models --epochs 200 --batch_size 32 --learning_rate 0.0002 --num_workers 48


In [ ]:
# ── 執行訓練（結果直接存進 Drive）─────────────────────────────────────
import os
os.makedirs(SAVE_PATH, exist_ok=True)
!{cmd_r34_train}

python3: can't open file '/content/src/train.py': [Errno 2] No such file or directory


## 9. 競賽二：ResNet34-UNet 推論（生成 submission.csv）

In [ ]:
# ── 10. ResNet34-UNet 推論設定 ────────────────────────────────────────
MODEL      = 'resnet34_unet'
DATA_PATH  = 'dataset/oxford-iiit-pet'
WEIGHT     = f'/content/drive/MyDrive/lab2/saved_models/{MODEL}_best.pth'
OUTPUT     = f'submission_{MODEL}.csv'
BATCH_SIZE = 32   #不需要跟訓練size一樣，只影響推論速度。  不要超過RAM限制就好

cmd_r34_inf = (f'python src/inference.py'
               f' --model {MODEL}'
               f' --data_path {DATA_PATH}'
               f' --weight {WEIGHT}'
               f' --output {OUTPUT}'
               f' --batch_size {BATCH_SIZE}')
print('Command:', cmd_r34_inf)


Command: python src/inference.py --model resnet34_unet --data_path dataset/oxford-iiit-pet --weight /content/drive/MyDrive/lab2/saved_models/resnet34_unet_best.pth --output submission_resnet34_unet.csv --batch_size 32


In [ ]:
# ── 執行推論 ──────────────────────────────────────────────────────────
!{cmd_r34_inf}

Device: cuda
Split dir: binary-semantic-segmentation-res-net-34-u-net
Loaded weights: /content/drive/MyDrive/lab2/saved_models/resnet34_unet_best.pth
Test samples: 739
Inference: 100% 24/24 [00:34<00:00,  1.43s/it]
Saved 739 predictions → submission_resnet34_unet.csv


In [ ]:
# ── 確認並下載 submission_resnet34_unet.csv ───────────────────────────
import pandas as pd
from google.colab import files

df = pd.read_csv(OUTPUT)
print(f'Submission shape: {df.shape}')
print(df.head())

# 存到 Drive
import shutil
shutil.copy(OUTPUT, f'/content/drive/MyDrive/lab2/{OUTPUT}')
print(f'Saved to Drive')

Submission shape: (739, 2)
              image_id                                       encoded_mask
0           Birman_109  14828 4 15321 28 15813 43 16312 47 16808 52 17...
1   english_setter_103  9000 1 9496 5 9992 9 10489 12 10985 16 11483 1...
2   great_pyrenees_138  14797 1 15019 8 15242 11 15467 12 15692 12 159...
3        pomeranian_19  34088 15 34442 15 34793 38 35141 52 35491 60 3...
4  scottish_terrier_89  6576 4 6944 12 7317 12 7684 20 8034 2 8040 39 ...
Saved to Drive


In [ ]:
# 下載到本機
files.download(OUTPUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. 停用 Google Colab GPU

In [ ]:
# ── 終止執行階段 ─────────────────────────────────────────────────────
from google.colab import runtime
runtime.unassign()